In [1]:
5

5

In [2]:
# !pip install pytorch-lightning

In [3]:
import torch
import pytorch_lightning as pl
from torchaudio.pipelines import HUBERT_BASE
import torchaudio
import numpy as np
import os
from torch.utils.data import DataLoader, Dataset, TensorDataset
from tqdm import tqdm
import gc
gc.collect()

/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/__init__.py:98: UserWarning: unable to load libtensorflow_io_plugins.so: unable to open file: libtensorflow_io_plugins.so, from paths: ['/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/libtensorflow_io_plugins.so']
caused by: ['/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/libtensorflow_io_plugins.so: undefined symbol: _ZN3tsl6StatusC1EN10tensorflow5error4CodeESt17basic_string_viewIcSt11char_traitsIcEENS_14SourceLocationE']
  warnings.warn(f"unable to load libtensorflow_io_plugins.so: {e}")
/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/__init__.py:104: UserWarning: file system plugins are not loaded: unable to open file: libtensorflow_io.so, from paths: ['/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/libtensorflow_io.so']
caused by: ['/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/libtensorflow_io.so: undefined symbol: _ZTVN10tenso

3169

In [4]:
import torch.nn as nn
import torch.nn.functional as F
import math
from dataclasses import dataclass, field

from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
from pytorch_lightning.callbacks import Callback, ModelCheckpoint
from pytorch_lightning.callbacks.progress.rich_progress import RichProgressBarTheme

import torchmetrics.functional as MF

In [5]:
train_100 = '/kaggle/input/librispeech-asr-wav-dataset/train-clean-100'
train_100 = list(map(lambda x: os.path.join(train_100, x), filter(lambda x: '.txt' not in x.lower(), os.listdir(train_100))))

train_300 = '/kaggle/input/librispeech-asr-wav-dataset/train-clean-360'
train_300 = list(map(lambda x: os.path.join(train_300, x), filter(lambda x: '.txt' not in x.lower(), os.listdir(train_300))))

train_500 = '/kaggle/input/librispeech-500-hours/LibriSpeech/train-other-500'
train_500 = list(map(lambda x: os.path.join(train_500, x), os.listdir(train_500)))
new_train_500 = []
for f in tqdm(train_500):
    ch = list(map(lambda x: os.path.join(f, x), filter(lambda x: '.txt' not in x.lower(),os.listdir(f))))
    for c in ch:
        flacs = list(map(lambda x: os.path.join(c, x), filter(lambda x: '.txt' not in x.lower(),os.listdir(c))))
        new_train_500.extend(flacs)
        
dev = '/kaggle/input/librispeech-asr-wav-dataset/dev-clean'
dev = list(map(lambda x: os.path.join(dev, x), filter(lambda x: '.txt' not in x.lower(), os.listdir(dev))))

train = train_100 + train_300 + new_train_500

100%|██████████| 1166/1166 [00:44<00:00, 26.33it/s]


In [6]:
class MuSTCDataset(Dataset):
    def __init__(self, data, max_size, frame_size, frame_stride):
        super(MuSTCDataset, self).__init__()
        self.max_size = max_size
        self.frame_size = frame_size
        self.frame_stride = frame_stride
        self.files = np.array(data)
        
    def  __getitem__(self, idx):
        data = self.files[idx]
        data, _ = torchaudio.load(data)
        
        if data.size(1) > self.max_size:
            data = data[:, :self.max_size]
        else:
            plus_len = self.max_size - data.size(1)
            data = F.pad(data, [0, plus_len], "constant", 0) 
        
#         data = self.__pad_wave(data)

#         data = data.unfold(dimension=1, size=self.frame_size, step=self.frame_stride).transpose(1, 0)
    
        return data
#     def __pad_wave(self, wave):
#         """Only supported shapes (B, C, Wave_size) or (B, Wave_size) or (Wave_size, ) for unbatched
#         """
        
#         B, C, w_len = 1, 1, None
#         if not isinstance(wave, torch.Tensor):
#             wave = torch.tensor(wave)
#         wave = wave
#         if wave.dim() == 3:
#             B, C, w_len = wave.size()
#         elif wave.dim() == 2:
#             B, w_len = wave.size()
#         elif wave.dim() == 1:
#             w_len, = wave.size()
#             wave = wave.unsqueeze(0)

#         else:
#             raise ValueError(f'expected number of dims is 1 for unbatched and 2 for batched but you provide {wave.dim()}')
    
#         assert w_len >= self.frame_size, f"vrey short wave lenght. minimum wave lenght is {self.frame_size}"
#         assert C == 1, f'only support mono wave at this time. Expected num of channels is {1} but given {C}'
        
#         wave = wave.reshape(C, w_len)                   
#         num_frames = int(math.ceil(float(abs(w_len - self.frame_size)) / self.frame_stride))
#         plus_len = (num_frames * self.frame_stride + self.frame_size) - w_len
#         wave = F.pad(wave, [0, plus_len], "constant", 0) 
#         return wave
    

    def __len__(self):
        return len(self.files)

In [7]:
class DataLightning(pl.LightningDataModule):
    def __init__(self, max_size, frame_size, frame_stride, batch_size):
        super().__init__()
        self.batch_size = batch_size
        self.max_size = max_size
        self.frame_size = frame_size
        self.frame_stride = frame_stride
    def train_dataloader(self):
        train_loader = MuSTCDataset(train, self.max_size, self.frame_size, self.frame_stride)

        train_loader = DataLoader(train_loader, batch_size=self.batch_size, shuffle=True, pin_memory=True, num_workers=2, persistent_workers=True)
        return train_loader 
    
    def val_dataloader(self):
        val_loader = MuSTCDataset(dev, self.max_size, self.frame_size, self.frame_stride)
        val_loader = DataLoader(val_loader, batch_size=self.batch_size, shuffle=False, prefetch_factor=None, pin_memory=True, num_workers=2, persistent_workers=False)
        return val_loader 


In [8]:
class Wave2Chunk(nn.Module):
    
    def __init__(self, frame_size, frame_stride, filters, out_dim=512):
        '''frame_size, frame_stride, filters, out_dim=512'''
        super(Wave2Chunk, self).__init__()
        self.frame_size, self.frame_stride, self.filters, self.out_dim = frame_size, frame_stride, filters, out_dim

        assert self.frame_size >= 2000, "the frame size minimum should be greater than or equal 2000"
        assert not self.frame_stride % self.filters, 'frame size should be divisible by num of fiinal chanels'
        
        pram_per_layer = [(11, 3, 0, 3), (11, 3, 0, 2), (2, 2, 0, 1),
                          (7, 1, 0, 1), (7, 2, 0, 1), (3, 3, 0, 1),
                          (5, 2, 0, 1), (5, 2, 0, 1), (2, 2, 0, 1),   
                          (3, 1, 0, 1), (3, 2, 0, 1), (2, 2, 0, 1)]
        
        out_shape = self.frame_size
        for p in pram_per_layer:
            out_shape = self._conv_output_length(out_shape, *p)
        
        self.conv_block_1 = nn.Sequential(nn.Conv1d(1, 2, 11, 3, dilation=3),  
                                          nn.ReLU(),
                                          nn.Conv1d(2, 4, 11, 3, dilation=2),  
                                          nn.ReLU(),
                                          nn.MaxPool1d(2, 2))
        
               
        self.conv_block_2 = nn.Sequential(nn.Conv1d(4, 8, 7, 1, dilation=1),  
                                          nn.ReLU(),
                                          nn.Conv1d(8, 12, 7, 2, dilation=1),  
                                          nn.ReLU(),
                                          nn.MaxPool1d(3, 3),                  
                                          nn.BatchNorm1d(12))
        
               
        self.conv_block_3 = nn.Sequential(nn.Conv1d(12, 16, 5, 2, dilation=1),  
                                          nn.ReLU(),
                                          nn.Conv1d(16, 22, 5, 2, dilation=1),  
                                          nn.ReLU(),
                                          nn.MaxPool1d(2, 2))
        
               
        self.conv_block_4 = nn.Sequential(nn.Conv1d(22, 32, 3, 1, dilation=1),  
                                          nn.ReLU(),
                                          nn.Conv1d(32, self.filters, 3, 2, dilation=1),  
                                          nn.ReLU(),
                                          nn.MaxPool1d(2, 2),                   
                                          nn.BatchNorm1d(self.filters))
        
        
        
        self.projection = nn.Sequential(nn.Linear(out_shape, self.out_dim * 2),
                                        nn.Dropout(0.01),
                                        nn.ReLU(),
                                        nn.Linear(self.out_dim * 2, self.out_dim))
        
        
    def _conv_output_length(self, length_in, kernel_size, stride=1, padding=0, dilation=1):
        return (length_in + 2 * padding - dilation * (kernel_size - 1) - 1) // stride + 1
    
    def forward(self, waves, mask=False, training=False):
        # ensure that all chunks will be the same size

        waves = self.__pad_wave(waves)
        
        waves = waves.unfold(dimension=2, size=self.frame_size, step=self.frame_stride).transpose(2, 1)
        
        B, chunks, c, seq_len = waves.size()
        
        waves = waves.contiguous().view(B*chunks, c, seq_len)
        
        waves = self.conv_block_1(waves)
        waves = self.conv_block_2(waves)
        waves = self.conv_block_3(waves)
        waves = self.conv_block_4(waves)
        waves = waves.view(B, -1, waves.size(-1)) #view(B * chunks * c, waves.size(-1))
        
        waves = self.projection(waves)
        return waves
        
        
    def __pad_wave(self, wave):
        """Only supported shapes (B, C, Wave_size) or (B, Wave_size) or (Wave_size, ) for unbatched
        """
        
        B, C, w_len = 1, 1, None
        if not isinstance(wave, torch.Tensor):
            wave = torch.tensor(wave)
        wave = wave
        if wave.dim() == 3:
            B, C, w_len = wave.size()
        elif wave.dim() == 2:
            B, w_len = wave.size()
        elif wave.dim() == 1:
            w_len, = wave.size()
            wave = wave.unsqueeze(0)

        else:
            raise ValueError(f'expected number of dims is 1 for unbatched and 2 for batched but you provide {wave.dim()}')
    
        assert w_len >= self.frame_size, f"vrey short wave lenght. minimum wave lenght is {self.frame_size}"
        assert C == 1, f'only support mono wave at this time. Expected num of channels is {1} but given {C}'
        
        wave = wave.reshape(B, C, w_len)                   
        num_frames = int(math.ceil(float(abs(w_len - self.frame_size)) / self.frame_stride))
        plus_len = (num_frames * self.frame_stride + self.frame_size) - w_len
        wave = F.pad(wave, [0, plus_len], "constant", 0) 
        return wave
    

In [9]:
class EncoderLayer(nn.Module):
    def __init__(self, d_model, nhead, batch_first=True):
        """@params:
                    d_model, nhead, batch_first=True
        """
        super(EncoderLayer, self).__init__()
        self.d_model, self.nhead, self.batch_first = d_model, nhead, batch_first      
        
        self.pre_norm = nn.LayerNorm(self.d_model)
        
        self.mha      = nn.MultiheadAttention(embed_dim=self.d_model, num_heads=self.nhead, dropout=0.01, 
                                               bias=True, add_bias_kv=True, batch_first=self.batch_first)
        

        self.ffn       = nn.Sequential(nn.LayerNorm(self.d_model),
                                       nn.Linear(self.d_model, 1080),
                                       nn.ReLU(),
                                       nn.Linear(1080, self.d_model),
                                       nn.ReLU())
        
        
        
    def forward(self, x, need_weights=False):
        if x.dim() < 3:
            x = x.view(1, -1, self.d_model).contiguous()
        x = self.pre_norm(x)
        att_out, att_weights = self.mha(query=x, key=x, value=x, need_weights=need_weights)

        att_out = att_out + x
        
        att_out = att_out + self.ffn(att_out)
        
        if need_weights:
            return att_out, att_weights
        
        return att_out
    

In [10]:
class Encoder(nn.Module):
    def __init__(self, window, stride,filters, d_model, n_head, size, batch_first=True):
        '''d_model=512, nhead=4, batch_first=True, size'''
        super().__init__()
        self.window, self.stride, self.filters, self.d_model, self.n_head, self.size, self.batch_first = window, stride, filters, d_model, n_head, size, batch_first
        
        assert self.d_model % self.n_head == 0, 'd_model should be dvisible by num of heads'
        self.feature_extractor = Wave2Chunk(frame_size=self.window, frame_stride=self.stride, filters=self.filters, out_dim=d_model)
        
        self.enc_stack = nn.ModuleList([EncoderLayer(d_model=self.d_model, nhead=self.n_head, batch_first=self.batch_first) 
                                        for _ in range(self.size)])
        
        
    def forward(self, x, need_weights=False):
        x = self.feature_extractor(x)
        if need_weights:
            
            for layer in self.enc_stack:
                x, atten_weights = layer(x, need_weights=need_weights)
            return x, atten_weights
        
        for layer in self.enc_stack:
                x = layer(x, need_weights=need_weights)
        return x

In [28]:
class Discremenator(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.d_model = d_model
        
        self.conv = nn.Sequential(nn.Conv1d(1, 2, 3,),
                                   nn.ReLU(),
                                   nn.Conv1d(2, 4, 3,),
                                   nn.ReLU(),
                                   nn.MaxPool1d(2, 3),
                                   nn.BatchNorm1d(4),
                                   nn.Conv1d(4, 8, 3,),
                                   nn.ReLU(),
                                   nn.Conv1d(8, 16, 3,),
                                   nn.LeakyReLU(0.2),
                                   nn.MaxPool1d(2),
                                   nn.BatchNorm1d(16),
                                   nn.Conv1d(16, 32, 3,),
                                   nn.ReLU(),
                                   nn.Conv1d(32, 64, 3,),
                                   nn.ReLU(),
                                   nn.MaxPool1d(2, 3),
                                   nn.BatchNorm1d(64),
                                   nn.Conv1d(64, 128, 3,),
                                   nn.ReLU(),
                                   nn.Conv1d(128, 256, 3,),
                                   nn.LeakyReLU(0.2),
                                   nn.MaxPool1d(2),
                                   nn.BatchNorm1d(256),
                                   nn.Flatten(),
                                   nn.Linear(1024, 2))
    def forward(self, x, **kw):
        B, S, D = x.size()
        x = x.reshape(-1, 1, self.d_model)
        x = self.conv(x)
        x = x.reshape(B, S, -1)
        return x

In [29]:
139.0 * 8

1112.0

In [30]:
class GenartiveTraining(pl.LightningModule):
    def __init__(self, window, stride,filters, d_model, n_head, size):
        super().__init__()
        self.save_hyperparameters()
        self.automatic_optimization = False 
        self.context = Encoder(self.hparams.window, self.hparams.stride, self.hparams.filters, self.hparams.d_model, self.hparams.n_head, self.hparams.size)
        
        self.detector = Discremenator(self.hparams.d_model)
        self.true_gen = HUBERT_BASE.get_model().requires_grad_(False)
        
    def forward(self, x, need_weights=False):
        return self.context(x, need_weights=need_weights)
    
    def training_step(self, batch, batch_idx):
        optimizer_g, optimizer_d = self.optimizers()
        # train generator
        self.detector.eval()
        self.context.train()
        self.toggle_optimizer(optimizer_g)
        # Forward Context
        g_hat  = self.context(batch)
        g_y = self.detector(g_hat)
        valid    = torch.ones(g_hat.size(0), g_hat.size(1), device=batch.device).long()
        g_loss = F.cross_entropy(torch.permute(g_y, (0, 2, 1)), valid)
        # generator optimizer step
        self.log("g_loss", g_loss, prog_bar=True)
        self.manual_backward(g_loss)
        self.clip_gradients(optimizer_g, gradient_clip_val=0.5, gradient_clip_algorithm="norm")
        optimizer_g.step()
        optimizer_g.zero_grad()
        self.untoggle_optimizer(optimizer_g)

        # train discriminator
        # Measure discriminator's ability to classify real from generated samples
        self.detector.train()
        self.context.eval()
        self.toggle_optimizer(optimizer_d)
        
        x_fake = self.context(batch)
        y_fake = torch.zeros(x_fake.size(0), x_fake.size(1), device=x_fake.device)
        
        x_true, _ = self.true_gen(batch.squeeze(1))
        x_true = F.adaptive_avg_pool2d(x_true, (x_fake.size(1), x_fake.size(2)))
        y_true = torch.ones(x_true.size(0), x_true.size(1), device=x_true.device)
        
        d_x_train = torch.cat([x_fake, x_true])
        d_y_train = torch.cat([y_fake, y_true]).long()
        
        x_train = TensorDataset(d_x_train, d_y_train)
        x_train = DataLoader(x_train, batch_size=batch.size(0), shuffle=True)
        jj = len(x_train)
        d_loss = 0
        for x, y in x_train:
            y_hat = self.detector(x)
            loss = F.cross_entropy(torch.permute(y_hat, (0, 2, 1)), y)
            d_loss += (loss / jj)
        self.manual_backward(d_loss)
        self.clip_gradients(optimizer_d, gradient_clip_val=0.5, gradient_clip_algorithm="norm")
        optimizer_d.step()
        optimizer_d.zero_grad()

        self.log("d_loss", d_loss / len(x_train), prog_bar=True)
        self.untoggle_optimizer(optimizer_d)
        
    def validation_step(self, batch, batch_idx):
        g_hat  = self.context(batch)
        g_y = self.detector(g_hat)
        valid    = torch.ones(g_hat.size(0), g_hat.size(1), device=batch.device).long()
        g_loss = F.cross_entropy(torch.permute(g_y, (0, 2, 1)), valid)
        # generator optimizer step
        self.log("g_loss", g_loss, prog_bar=True)
        
        x_fake = self.context(batch)
        y_fake = torch.zeros(x_fake.size(0), x_fake.size(1), device=x_fake.device)
        
        x_true, _ = self.true_gen(batch.squeeze(1))
        x_true = F.adaptive_avg_pool2d(x_true, (x_fake.size(1), x_fake.size(2)))
        y_true = torch.ones(x_true.size(0), x_true.size(1), device=x_true.device)
        
        d_x_train = torch.cat([x_fake, x_true])
        d_y_train = torch.cat([y_fake, y_true]).long()
        
        x_train = TensorDataset(d_x_train, d_y_train)
        x_train = DataLoader(x_train, batch_size=batch.size(0), shuffle=True)
        d_loss = 0
        jj = len(x_train)
        for x, y in x_train:
            y_hat = self.detector(x)
            loss = F.cross_entropy(torch.permute(y_hat, (0, 2, 1)), y)
            d_loss += (loss / jj)
            

        self.log("d_loss", d_loss / len(x_train), prog_bar=True)
        
    def on_train_epoch_end(self, **kw):
        sch = self.lr_schedulers()
        sch.step()

    def init_weights(self):
        
        for p in self.context.parameters():    
            if p.dim() != 1:
                nn.init.xavier_normal_(p.data)
            else:
                nn.init.zeros_(p.data)
                
        for m in self.context.modules():
            if isinstance(m, (nn.LayerNorm, nn.BatchNorm1d)):
                m.reset_parameters()
                
        for p in self.detector.parameters():    
            if p.dim() != 1:
                nn.init.xavier_normal_(p.data)
            else:
                nn.init.zeros_(p.data)
                
        for m in self.detector.modules():
            if isinstance(m, (nn.LayerNorm, nn.BatchNorm1d)):
                m.reset_parameters()

    def configure_optimizers(self):
        opt_g = torch.optim.SGD(self.context.parameters(), lr=8e-3, momentum=0.4, weight_decay=0.1)
        
        opt_d = torch.optim.AdamW(self.detector.parameters(), lr=2e-3, betas=(0.5, 0.999), weight_decay=0.4)
        scheduler = {
            "scheduler": CosineAnnealingWarmRestarts(opt_g, T_0=8, T_mult=2, eta_min=5e-5, verbose=True)}
        return [opt_g, opt_d], [scheduler]        

In [31]:
ckp = ModelCheckpoint(every_n_train_steps=4000, save_last=True, auto_insert_metric_name=False,
                      dirpath='/kaggle/working/lightning_logs/checkpoints')

In [32]:
data_loader = DataLightning(15*16000, 16000*3, 16000*2, 16)
hyper_parameter = dict(window=16000*3, stride=16000*2,filters=25, d_model=280, n_head=14, size=30)

model = GenartiveTraining(**hyper_parameter)

In [33]:
pl.seed_everything(23, workers=True)
worker = 2
accelerator = 'auto'
devices = 1
strategy = 'auto'
epochs = 10000

In [34]:
# import os
# import shutil 

# root = '/kaggle/input/teacher-student-for-wave-encoding/lightning_logs/'
# for d in os.listdir(root):
#     if 'checkpoints' in d:
#         continue
#     dest = os.path.join('lightning_logs', d)
#     os.makedirs(dest, exist_ok=True)
#     for f in os.listdir(os.path.join(root, d)):
#         shutil.copy2(os.path.join(root, d, f), dest)

In [ ]:
trainer = pl.Trainer(accelerator=accelerator, devices=1, 
                     max_epochs=epochs,
                     strategy=strategy,
                     num_sanity_val_steps=4,
                     log_every_n_steps=400,
                     callbacks=[ ckp],
#                      accumulate_grad_batches=2,
                     sync_batchnorm=True,
                     enable_model_summary=True, enable_checkpointing=True, # benchmark=True, 
                     default_root_dir='/kaggle/working/')

trainer.fit(model, data_loader)#, ckpt_path='/kaggle/input/teacher-student-for-wave-encoding/lightning_logs/checkpoints/last.ckpt')

Epoch 00000: adjusting learning rate of group 0 to 8.0000e-03.


Sanity Checking: 0it [00:00, ?it/s]

Training: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

Epoch 00001: adjusting learning rate of group 0 to 7.6974e-03.


In [ ]:
def conv_output_length(length_in, kernel_size, stride=1, padding=0, dilation=1):
        return (length_in + 2 * padding - dilation * (kernel_size - 1) - 1) // stride + 1
    

In [ ]:
138

In [ ]:
conv_output_length(251, 2 , 2)

In [ ]:

nn.Conv1d(4, 8, 3,),
nn.ReLU(),
nn.Conv1d(8, 16, 3,),
nn.ReLU(),
nn.MaxPool1d(2),

In [ ]:
!rm -r lightning_logs/